# Multi-Protocol PCAP Feature Extraction for Anomaly Detection

This notebook extracts network packet fields from PCAP files using tshark and exports them to CSV for ML-based anomaly detection. Supports multiple protocols including:
- **Modbus/TCP** - Industrial control protocol
- **Generic TCP** - All other TCP traffic
- **UDP** - User Datagram Protocol traffic
- **ICMP** - Internet Control Message Protocol

Each packet includes a `protocol` column identifying the detected application-layer protocol.

In [95]:
# Requires tshark (Wireshark CLI) to be installed and in PATH
# Download from: https://www.wireshark.org/download.html

In [96]:
import subprocess
import json
import pandas as pd
from datetime import datetime
from pathlib import Path
import csv
import os

In [97]:
def extract_packet_features(pcap_path: str, protocol_filter: str = None) -> list[dict]:
    """
    Extract packet fields from a PCAP file using tshark.
    Supports multiple protocols: Modbus/TCP, generic TCP, UDP, ICMP.
    
    Args:
        pcap_path: Path to the PCAP file
        protocol_filter: Optional tshark display filter (e.g., "modbus", "tcp", "udp")
        
    Returns:
        List of dictionaries containing extracted features
    """
    # Define fields to extract for all protocols
    fields = [
        # Frame fields
        "frame.number",
        "frame.time_epoch",
        "frame.len",
        "frame.protocols",
        # IP fields
        "ip.src",
        "ip.dst",
        "ip.proto",
        "ip.ttl",
        "ip.len",
        # TCP fields
        "tcp.srcport",
        "tcp.dstport",
        "tcp.len",
        "tcp.flags",
        "tcp.flags.syn",
        "tcp.flags.ack",
        "tcp.flags.fin",
        "tcp.flags.reset",
        "tcp.flags.push",
        "tcp.seq",
        "tcp.ack",
        "tcp.window_size",
        # UDP fields
        "udp.srcport",
        "udp.dstport",
        "udp.length",
        # ICMP fields
        "icmp.type",
        "icmp.code",
        # Modbus/TCP fields
        "mbtcp.trans_id",
        "mbtcp.prot_id",
        "mbtcp.len",
        "mbtcp.unit_id",
        "modbus.func_code",
        "modbus.reference_num",
        "modbus.word_cnt",
        "modbus.bit_cnt",
        "modbus.byte_cnt",
        "modbus.exception_code",
        # HTTP fields (common protocol)
        "http.request.method",
        "http.response.code",
        "http.host",
        # DNS fields
        "dns.qry.name",
        "dns.qry.type",
        "dns.flags.response",
    ]
    
    # Build tshark command
    cmd = [
        "tshark",
        "-r", str(pcap_path),
        "-T", "fields",
        "-E", "header=y",
        "-E", "separator=|",
        "-E", "quote=d",
    ]
    
    # Add optional filter
    if protocol_filter:
        cmd.extend(["-Y", protocol_filter])
    
    # Add field arguments
    for field in fields:
        cmd.extend(["-e", field])
    
    # Run tshark
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        raise RuntimeError(f"tshark failed: {result.stderr}")
    
    # Parse output
    lines = result.stdout.strip().split("\n")
    if len(lines) < 2:
        return []
    
    headers = lines[0].split("|")
    records = []
    
    def parse_int(val):
        """Parse int, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return int(val)
        except ValueError:
            return None
    
    def parse_float(val):
        """Parse float, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return float(val)
        except ValueError:
            return None
    
    def parse_str(val):
        """Parse string, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        return val or None
    
    def detect_protocol(row):
        """Detect the application-layer protocol from frame.protocols and fields."""
        protocols_str = parse_str(row.get('frame.protocols', ''))
        
        # Check for specific application protocols
        if protocols_str:
            if 'modbus' in protocols_str.lower() or 'mbtcp' in protocols_str.lower():
                return 'modbus'
            if 'http' in protocols_str.lower():
                return 'http'
            if 'dns' in protocols_str.lower():
                return 'dns'
            if 'tls' in protocols_str.lower() or 'ssl' in protocols_str.lower():
                return 'tls'
        
        # Check based on well-known ports if no app protocol detected
        src_port = parse_int(row.get('tcp.srcport')) or parse_int(row.get('udp.srcport'))
        dst_port = parse_int(row.get('tcp.dstport')) or parse_int(row.get('udp.dstport'))
        
        # Modbus port
        if src_port == 502 or dst_port == 502:
            return 'modbus'
        
        # Fall back to transport protocol
        ip_proto = parse_int(row.get('ip.proto'))
        if row.get('tcp.srcport') or row.get('tcp.dstport'):
            return 'tcp'
        if row.get('udp.srcport') or row.get('udp.dstport'):
            return 'udp'
        if row.get('icmp.type'):
            return 'icmp'
        if ip_proto == 6:
            return 'tcp'
        if ip_proto == 17:
            return 'udp'
        if ip_proto == 1:
            return 'icmp'
        
        return 'other'
    
    for line in lines[1:]:
        values = line.split("|")
        row = dict(zip(headers, values))
        
        timestamp = parse_float(row.get('frame.time_epoch'))
        detected_protocol = detect_protocol(row)
        
        # Base record with common fields
        record = {
            'packet_number': parse_int(row.get('frame.number')),
            'timestamp': timestamp,
            'datetime': datetime.fromtimestamp(timestamp).isoformat() if timestamp else None,
            'protocol': detected_protocol,
            'frame_protocols': parse_str(row.get('frame.protocols')),
            'src_ip': parse_str(row.get('ip.src')),
            'dst_ip': parse_str(row.get('ip.dst')),
            'ip_proto': parse_int(row.get('ip.proto')),
            'ip_ttl': parse_int(row.get('ip.ttl')),
            'ip_len': parse_int(row.get('ip.len')),
            'pkt_len': parse_int(row.get('frame.len')),
        }
        
        # TCP fields
        if detected_protocol in ('tcp', 'modbus', 'http', 'tls'):
            record.update({
                'src_port': parse_int(row.get('tcp.srcport')),
                'dst_port': parse_int(row.get('tcp.dstport')),
                'tcp_len': parse_int(row.get('tcp.len')),
                'tcp_flags': parse_str(row.get('tcp.flags')),
                'tcp_syn': parse_int(row.get('tcp.flags.syn')),
                'tcp_ack': parse_int(row.get('tcp.flags.ack')),
                'tcp_fin': parse_int(row.get('tcp.flags.fin')),
                'tcp_rst': parse_int(row.get('tcp.flags.reset')),
                'tcp_push': parse_int(row.get('tcp.flags.push')),
                'tcp_seq': parse_int(row.get('tcp.seq')),
                'tcp_ack_num': parse_int(row.get('tcp.ack')),
                'tcp_window': parse_int(row.get('tcp.window_size')),
            })
        
        # UDP fields
        if detected_protocol in ('udp', 'dns'):
            record.update({
                'src_port': parse_int(row.get('udp.srcport')),
                'dst_port': parse_int(row.get('udp.dstport')),
                'udp_len': parse_int(row.get('udp.length')),
            })
        
        # ICMP fields
        if detected_protocol == 'icmp':
            record.update({
                'icmp_type': parse_int(row.get('icmp.type')),
                'icmp_code': parse_int(row.get('icmp.code')),
            })
        
        # Modbus-specific fields
        if detected_protocol == 'modbus':
            record.update({
                'transaction_id': parse_int(row.get('mbtcp.trans_id')),
                'modbus_protocol_id': parse_int(row.get('mbtcp.prot_id')),
                'modbus_length': parse_int(row.get('mbtcp.len')),
                'unit_id': parse_int(row.get('mbtcp.unit_id')),
                'function_code': parse_int(row.get('modbus.func_code')),
                'reference_num': parse_int(row.get('modbus.reference_num')),
                'word_count': parse_int(row.get('modbus.word_cnt')),
                'bit_count': parse_int(row.get('modbus.bit_cnt')),
                'byte_count': parse_int(row.get('modbus.byte_cnt')),
                'exception_code': parse_int(row.get('modbus.exception_code')),
                'is_request': 1 if record.get('dst_port') == 502 else 0,
                'is_response': 1 if record.get('src_port') == 502 else 0,
                'is_exception': 1 if parse_int(row.get('modbus.exception_code')) is not None else 0,
            })
        
        # HTTP fields
        if detected_protocol == 'http':
            record.update({
                'http_method': parse_str(row.get('http.request.method')),
                'http_response_code': parse_int(row.get('http.response.code')),
                'http_host': parse_str(row.get('http.host')),
            })
        
        # DNS fields
        if detected_protocol == 'dns':
            record.update({
                'dns_query_name': parse_str(row.get('dns.qry.name')),
                'dns_query_type': parse_int(row.get('dns.qry.type')),
                'dns_is_response': parse_int(row.get('dns.flags.response')),
            })
        
        records.append(record)
    
    return records


# Keep backward compatibility
def extract_modbus_features(pcap_path: str) -> list[dict]:
    """Extract Modbus/TCP packets only (backward compatible)."""
    return extract_packet_features(pcap_path, protocol_filter="modbus")

In [98]:
# Modbus function code reference for labeling
MODBUS_FUNCTION_CODES = {
    1: 'Read Coils',
    2: 'Read Discrete Inputs',
    3: 'Read Holding Registers',
    4: 'Read Input Registers',
    5: 'Write Single Coil',
    6: 'Write Single Register',
    7: 'Read Exception Status',
    8: 'Diagnostics',
    15: 'Write Multiple Coils',
    16: 'Write Multiple Registers',
    22: 'Mask Write Register',
    23: 'Read/Write Multiple Registers',
    43: 'Read Device Identification',
}

def add_function_name(df: pd.DataFrame) -> pd.DataFrame:
    """Add human-readable function code names for Modbus packets."""
    if 'function_code' in df.columns:
        df['function_name'] = df['function_code'].map(MODBUS_FUNCTION_CODES).fillna('Unknown')
    return df

In [99]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add time-based features useful for anomaly detection.
    """
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # Inter-arrival time
    df['inter_arrival_time'] = df['timestamp'].diff()
    
    # Time since first packet
    df['time_from_start'] = df['timestamp'] - df['timestamp'].iloc[0]
    
    # Rolling statistics (last 10 packets)
    df['rolling_iat_mean'] = df['inter_arrival_time'].rolling(window=10, min_periods=1).mean()
    df['rolling_iat_std'] = df['inter_arrival_time'].rolling(window=10, min_periods=1).std()
    
    return df

In [100]:
def add_flow_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add flow-based features for each unique connection pair.
    """
    # Create flow identifier
    df['flow_id'] = df.apply(
        lambda x: f"{min(str(x['src_ip']), str(x['dst_ip']))}_{max(str(x['src_ip']), str(x['dst_ip']))}",
        axis=1
    )
    
    # Packet count per flow
    df['flow_pkt_count'] = df.groupby('flow_id').cumcount() + 1
    
    # Function code diversity per flow (rolling)
    df['unique_func_codes'] = df.groupby('flow_id')['function_code'].transform(
        lambda x: x.expanding().apply(lambda y: y.nunique())
    )
    
    return df

## Usage Example

In [101]:
# === CONFIGURE YOUR PCAP FILE PATH HERE ===
relative_path = "data\\raw\\captures1_v2\\clean\\eth2dump-clean-1h_1.pcap"
absolute_path = (Path("..") / relative_path).resolve()

print(f"Using PCAP file at: {absolute_path}")

PCAP_PATH = str(absolute_path)  # Convert to string for tshark
OUTPUT_CSV = "packet_features.csv"

# Optional: Set protocol filter (None = all protocols, "modbus" = only Modbus, "tcp" = only TCP, etc.)
PROTOCOL_FILTER = None  # Change to "modbus" for Modbus-only extraction

Using PCAP file at: C:\Users\jorel\OneDrive\Documents\CODE STUFF\DOE\BNL\foundational_model_for_energy_security\data\raw\captures1_v2\clean\eth2dump-clean-1h_1.pcap


In [102]:
# Extract packet features (all protocols or filtered)
print(f"Processing: {PCAP_PATH}")
print(f"Protocol filter: {PROTOCOL_FILTER or 'All protocols'}")

records = extract_packet_features(PCAP_PATH, protocol_filter=PROTOCOL_FILTER)
print(f"Extracted {len(records)} packets")

Processing: C:\Users\jorel\OneDrive\Documents\CODE STUFF\DOE\BNL\foundational_model_for_energy_security\data\raw\captures1_v2\clean\eth2dump-clean-1h_1.pcap
Protocol filter: All protocols
Extracted 72150 packets


In [103]:
# Convert to DataFrame and add derived features
df = pd.DataFrame(records)

if len(df) > 0:
    df = add_function_name(df)
    df = add_temporal_features(df)
    df = add_flow_features(df)
    
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
else:
    print("No Modbus packets found in PCAP!")


DataFrame shape: (72150, 45)

Columns: ['packet_number', 'timestamp', 'datetime', 'protocol', 'frame_protocols', 'src_ip', 'dst_ip', 'ip_proto', 'ip_ttl', 'ip_len', 'pkt_len', 'src_port', 'dst_port', 'tcp_len', 'tcp_flags', 'tcp_syn', 'tcp_ack', 'tcp_fin', 'tcp_rst', 'tcp_push', 'tcp_seq', 'tcp_ack_num', 'tcp_window', 'transaction_id', 'modbus_protocol_id', 'modbus_length', 'unit_id', 'function_code', 'reference_num', 'word_count', 'bit_count', 'byte_count', 'exception_code', 'is_request', 'is_response', 'is_exception', 'udp_len', 'function_name', 'inter_arrival_time', 'time_from_start', 'rolling_iat_mean', 'rolling_iat_std', 'flow_id', 'flow_pkt_count', 'unique_func_codes']


In [104]:
# Preview the data
df.head(10)

,packet_number,timestamp,datetime,protocol,frame_protocols,src_ip,dst_ip,ip_proto,ip_ttl,ip_len,...,is_exception,udp_len,function_name,inter_arrival_time,time_from_start,rolling_iat_mean,rolling_iat_std,flow_id,flow_pkt_count,unique_func_codes
0,1,1.536440e+09,2018-09-08T16:54:30.055515,other,eth:llc:stp,None,None,NaN,NaN,NaN,...,NaN,NaN,Unknown,NaN,0.000000,NaN,NaN,None_None,1,NaN
1,2,1.536440e+09,2018-09-08T16:54:30.111272,modbus,eth:ethertype:ip:tcp,172.27.224.70,172.27.224.250,6.0,128.0,40.0,...,0.0,NaN,Unknown,0.055757,0.055757,0.055757,NaN,172.27.224.250_172.27.224.70,1,NaN
2,3,1.536440e+09,2018-09-08T16:54:30.205937,modbus,eth:ethertype:ip:tcp:mbtcp:modbus,172.27.224.70,172.27.224.250,6.0,128.0,52.0,...,0.0,NaN,Read Holding Registers,0.094665,0.150422,0.075211,0.027512,172.27.224.250_172.27.224.70,2,1.0
3,4,1.536440e+09,2018-09-08T16:54:30.209039,modbus,eth:ethertype:ip:tcp:mbtcp:modbus,172.27.224.250,172.27.224.70,6.0,64.0,71.0,...,0.0,NaN,Read Holding Registers,0.003102,0.153524,0.051175,0.045953,172.27.224.250_172.27.224.70,3,1.0
4,5,1.536440e+09,2018-09-08T16:54:30.423275,modbus,eth:ethertype:ip:tcp,172.27.224.70,172.27.224.250,6.0,128.0,40.0,...,0.0,NaN,Unknown,0.214236,0.367760,0.091940,0.089750,172.27.224.250_172.27.224.70,4,1.0
5,6,1.536440e+09,2018-09-08T16:54:30.517529,modbus,eth:ethertype:ip:tcp:mbtcp:modbus,172.27.224.70,172.27.224.250,6.0,128.0,52.0,...,0.0,NaN,Read Holding Registers,0.094254,0.462014,0.092403,0.077733,172.27.224.250_172.27.224.70,5,1.0
6,7,1.536440e+09,2018-09-08T16:54:30.528387,modbus,eth:ethertype:ip:tcp:mbtcp:modbus,172.27.224.250,172.27.224.70,6.0,64.0,71.0,...,0.0,NaN,Read Holding Registers,0.010858,0.472872,0.078812,0.077085,172.27.224.250_172.27.224.70,6,1.0
7,8,1.536440e+09,2018-09-08T16:54:30.735268,modbus,eth:ethertype:ip:tcp,172.27.224.70,172.27.224.250,6.0,128.0,40.0,...,0.0,NaN,Unknown,0.206881,0.679753,0.097108,0.085410,172.27.224.250_172.27.224.70,7,1.0
8,9,1.536440e+09,2018-09-08T16:54:30.751871,udp,eth:ethertype:ip:udp:data,172.27.224.70,255.255.255.255,17.0,128.0,68.0,...,NaN,48.0,Unknown,0.016603,0.696356,0.087045,0.084041,172.27.224.70_255.255.255.255,1,NaN
9,10,1.536440e+09,2018-09-08T16:54:30.829737,modbus,eth:ethertype:ip:tcp:mbtcp:modbus,172.27.224.70,172.27.224.250,6.0,128.0,52.0,...,0.0,NaN,Read Holding Registers,0.077866,0.774222,0.086025,0.078673,172.27.224.250_172.27.224.70,8,1.0


In [105]:
# Summary statistics
print("Protocol Distribution:")
print(df['protocol'].value_counts())

# Show Modbus function codes if any Modbus packets exist
if 'function_name' in df.columns and df['protocol'].eq('modbus').any():
    print("\nModbus Function Code Distribution:")
    modbus_df = df[df['protocol'] == 'modbus']
    print(modbus_df['function_name'].value_counts())

print("\nBasic Statistics:")
numeric_cols = ['pkt_len', 'inter_arrival_time', 'ip_len']
available_cols = [c for c in numeric_cols if c in df.columns]
if available_cols:
    print(df[available_cols].describe())

Protocol Distribution:
protocol
modbus    65601
other      5788
udp         761
Name: count, dtype: int64

Modbus Function Code Distribution:
function_name
Unknown                   41356
Read Holding Registers    23003
Write Single Register      1242
Name: count, dtype: int64

Basic Statistics:
            pkt_len  inter_arrival_time        ip_len
count  72150.000000        7.214900e+04  66299.000000
mean      67.476466        4.989417e-02     50.210214
std       26.774419        6.951043e-02     27.428763
min       60.000000        9.536743e-07     40.000000
25%       60.000000        2.598763e-04     40.000000
50%       60.000000        8.908033e-03     41.000000
75%       66.000000        9.427381e-02     52.000000
max     1059.000000        1.094676e+00   1025.000000


In [106]:
# Export to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved to: {OUTPUT_CSV}")


Saved to: packet_features.csv


## Feature Summary for Anomaly Detection

### Common Fields (All Protocols)
| Feature | Description |
|---------|-------------|
| `protocol` | Detected protocol (modbus, tcp, udp, icmp, http, dns, tls, other) |
| `frame_protocols` | Full protocol stack from Wireshark |
| `src_ip`, `dst_ip` | Source and destination IP addresses |
| `src_port`, `dst_port` | Source and destination ports |
| `pkt_len`, `ip_len` | Packet and IP payload lengths |
| `ip_ttl` | IP Time-To-Live |

### TCP-Specific Fields
| Feature | Description |
|---------|-------------|
| `tcp_flags`, `tcp_syn/ack/fin/rst/push` | TCP flags for connection state |
| `tcp_seq`, `tcp_ack_num` | Sequence and acknowledgment numbers |
| `tcp_window` | TCP window size |

### Modbus-Specific Fields
| Feature | Description |
|---------|-------------|
| `function_code`, `function_name` | Modbus command type |
| `unit_id`, `transaction_id` | Device and transaction identifiers |
| `reference_num`, `word_count` | Register addressing |
| `is_request`, `is_response`, `is_exception` | Packet classification |

### Derived Features
| Feature Category | Fields | Use Case |
|-----------------|--------|----------|
| **Temporal** | `inter_arrival_time`, `rolling_iat_*` | Detect timing anomalies, DoS |
| **Flow** | `flow_pkt_count`, `unique_func_codes` | Detect reconnaissance, scanning |